# First Pass Annotation of cell types with reasonable resolution of annotation.
* Leiden_vi cluster is considered as baseline.
* Differential expression between clusters is performed using sc.tl.rank_gene_groups.
* Significantly differentially expressed genes with positive logfoldchanges values are arranged with respect to scores (descending).
* Top 30 genes are considered as marker genes and considered for manual annotation.
* The DE csv file saved, the txt file with genes considered to make prediction saved.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotting_utils

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad')

In [ ]:
adata

In [ ]:
plotting_utils.plot_umap(adata, color = 'leiden_vi', s1=10, s2=8, size=10, out=True)
# plotting_utils.plot_umap(adata, color = 'annot_1', s1=10, s2=8, size=10, out=True)

In [ ]:
plotting_utils.plot_umap(adata, color = 'annot_2', s1=10, s2=8, size=10, out=True)

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_vi', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes = 25, sharey = False)
df = sc.get.rank_genes_groups_df(adata, None)
df

In [ ]:
# Filter and separate each group
# Get unique groups
groups = df['group'].unique()

# Store results for each group
results = {}

for group in groups:
    # Filter for this group
    group_df = df[df['group'] == group].copy()
    
    # Apply filters: pvals_adj < 0.05 and (logfoldchanges > 1 OR logfoldchanges is NaN)
    filtered_df = group_df[
        (group_df['pvals_adj'] < 0.05) & 
        ((group_df['logfoldchanges'] > 1) | (group_df['logfoldchanges'].isna()))
    ].copy()
    
    # Sort by score (highest first)
    filtered_df = filtered_df.sort_values('scores', ascending=False)
    
    # Get top 30 genes
    top_30 = filtered_df.head(30)
    
    results[group] = {
        'top_30': top_30,
        'total_filtered': len(filtered_df)
    }

# Print all gene lists first
print("="*60)
print("GENE LISTS FOR ANNOTATION (TOP 30 PER GROUP)")
print("="*60)

for group in groups:
    print(f"\nGROUP {group} ({results[group]['total_filtered']} genes passing filters):")
    print(', '.join(results[group]['top_30']['names'].tolist()))

# Then print detailed tables
print("\n\n" + "="*60)
print("DETAILED TABLES")
print("="*60)

for group in groups:
    print(f"\n{'='*60}")
    print(f"GROUP {group} - Top 30 Marker Genes")
    print(f"{'='*60}\n")
    print(results[group]['top_30'][['names', 'scores', 'logfoldchanges', 'pvals_adj']].to_string(index=False))

In [ ]:
annotation = {
    '0':'Myoepithelial cells - tumor',
    '1':'Epithelial cells - tumor',
    '2':'Cycling cells - tumor',
    '3':'Fibroblast cells',
    '4':'Tumor Associated Macrophage cells',
    '5':'Unassigned Mixed (putative doublets)',
    '6':'Epithelial cells - tumor',
    '7':'T cells',
    '8':'Epithelial cells - tumor',
    '9':'Endothelial cells',
    '10':'Epithelial cells - tumor',
    '11':'Epithelial cells - tumor',
    '12':'Myoepithelial cells - tumor',
    '13':'Epithelial cells - tumor',
    '14':'Epithelial cells - tumor',
    '15':'Tongue muscle cells',
    '16':'Myoepithelial cells - tumor',
    '17':'Vascular smooth muscle cells',
    '18':'Vascular smooth muscle cells',
    '19':'B cells - Plasma cells',
    '20':'B cells',
    '21':'Basal cells - tumor',
    '22':'Ciliated epithelial cells',
    '23':'Mast cells',
    '24':'Myeloid cells – osteoclast-like macrophages'
}

In [ ]:
plotting_utils.plot_umap(adata, color = 'leiden_vi', s1=10, s2=8, size=10, out=True)
plotting_utils.plot_umap(adata, color = 'annot_2', s1=10, s2=8, size=10, out=True)

In [ ]:
adata.obs['annot_3'] = adata.obs['leiden_vi'].map(annotation)

In [ ]:
plotting_utils.plot_umap(adata, color = 'annot_3', s1=10, s2=8, size=10, out=True)

In [ ]:
# df.to_csv(f'{ACC_DATA_ROOT}/outputs/csv_files/annotation_2/df_test.csv')

In [ ]:
# adata.write_h5ad(f'{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad', compression='lzf')

In [ ]:
plotting_utils.plot_umap(adata, color = 'leiden_vi', s1=12, s2=10, size=12, out=False)
plt.savefig(f'{ACC_DATA_ROOT}/outputs/images/leiden_vi_clusters.pdf',
            format='pdf',
            dpi = 600,
            bbox_inches = 'tight')
plt.close()

In [ ]:
plotting_utils.plot_umap(adata, color = 'annot_3', s1=12, s2=10, size=12, out=False)
plt.savefig(f'{ACC_DATA_ROOT}/outputs/images/annot_3_clusters.pdf',
            format='pdf',
            dpi = 600,
            bbox_inches = 'tight')
plt.close()